[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day31-awq-quantization.ipynb)

In [ ]:
# Cell 1: one-time installs. Requires a T4 GPU runtime (Runtime > Change runtime type).
!pip install --quiet autoawq transformers accelerate datasets
# Expected: installs in ~2-3 min, no errors.

## Part 1 — The scale-and-protect trick by hand (CPU, ~5 min)

A single channel. Weights are tiny (plus/minus 0.01) but its activation is
100x larger than the rest. We compare plain round-to-nearest int4 against
AWQ's trick: scale weights by s=2, halve the activation, quantize, then
measure the output error. fp16 math: (W*s)(X/s) = W*X exactly — only the
quantization grid placement changes.

Expected: plain RTN destroys the channel (rounds to zero); the s=2 version
keeps ~2x finer effective grid and roughly halves the output error.

In [ ]:
import numpy as np

np.random.seed(31)
# Salient channel: 16 tiny weights, activation 100x the others
W = np.random.uniform(-0.01, 0.01, size=16)
X_normal, X_salient = 1.0, 100.0

def quantize_int4_per_channel(w, s=1.0):
    ws = w * s                      # AWQ: scale weights BEFORE quantizing
    scale = np.max(np.abs(ws)) / 7  # int4 symmetric: 15 levels -> [-7,7]
    q = np.round(ws / scale).clip(-8, 7)
    w_hat = q * scale / s           # dequantize back to original units
    return w_hat

# Plain RTN (s = 1)
w_rtn = quantize_int4_per_channel(W, s=1.0)
err_rtn = np.abs((w_rtn - W) * X_salient).sum()
print("RTN:   weights after quant:", np.unique(np.round(w_rtn, 4)))
print(f"RTN:   output error = {err_rtn:.4f}")

# AWQ (s = 2)
w_awq = quantize_int4_per_channel(W, s=2.0)
err_awq = np.abs((w_awq - W) * X_salient).sum()
print("AWQ s=2: distinct levels used:", len(np.unique(np.round(w_awq, 5))))
print(f"AWQ s=2: output error = {err_awq:.4f}")
print(f"Error ratio (RTN/AWQ) = {err_rtn/max(err_awq,1e-9):.2f}x")
# Expected: RTN output error several x larger than AWQ; RTN collapses weights to 0.

## Part 2 — Real AWQ on Qwen2.5-1.5B (T4, ~15-30 min)

Same model as Day 30, same calibration data, different algorithm.
Compare today's numbers against yesterday's fp16 / GPTQ table.

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "- free GB:", round(torch.cuda.mem_get_info()[0]/1e9, 1))
# Expected: cuda available: True, Tesla T4, ~14 GB free.

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("vocab size:", tokenizer.vocab_size)
# Expected: vocab size: 152064

In [ ]:
from awq import AutoAWQForCausalLM
import time

# fp16 in VRAM first (~3 GB), then quantize in place
model = AutoAWQForCausalLM.from_pretrained(model_id, safetensors=True, device_map="auto")
print("loaded fp16 OK")
t0 = time.time()
model.quantize(tokenizer, quant_config={"zero_point": True, "q_group_size": 128, "w_bit": 4})
print(f"AWQ quantize wall time: {(time.time()-t0)/60:.1f} min")
# Expected: loads in ~1-2 min, quantize ~5-15 min on T4, no OOM.

In [ ]:
import math
from torch.nn import CrossEntropyLoss
from datasets import load_dataset

def perplexity(m, tokenizer, n_seqs=50, seqlen=512):
    test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    enc = tokenizer("\n\n".join(test["text"]), return_tensors="pt")
    seq_len = enc.input_ids.size(1)
    nlls = []
    for i in range(0, min(n_seqs * seqlen, seq_len - seqlen), seqlen):
        batch = enc.input_ids[:, i:i+seqlen].to(m.device if hasattr(m, "device") else "cuda")
        with torch.no_grad():
            logits = m(batch).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = batch[:, 1:].contiguous()
        loss = CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)),
                                  shift_labels.view(-1))
        nlls.append(loss)
    return math.exp(torch.stack(nlls).mean().item())

ppl_awq = perplexity(model.model, tokenizer)
print(f"AWQ 4-bit g128 perplexity: {ppl_awq:.2f}")
print("Compare: Day 30 fp16 ~8-10, GPTQ delta < 0.3. Expect AWQ delta <= GPTQ delta.")
# Expected: ppl within ~0.25 of the fp16 baseline you measured on Day 30.

In [ ]:
# Save, reload the quantized checkpoint, and sanity-generate 50 tokens
model.save_quantized("./qwen2.5-1.5b-awq-4bit")
reloaded = AutoAWQForCausalLM.from_quantized("./qwen2.5-1.5b-awq-4bit", device_map="auto")
inp = tokenizer("The future of AI inference is", return_tensors="pt").to(reloaded.device)
out = reloaded.generate(**inp, max_new_tokens=50, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))
# Expected: coherent continuation, no repeated-token loops.

In [ ]:
import os
size_gb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk("./qwen2.5-1.5b-awq-4bit") for f in fs) / 1e9
print(f"checkpoint on disk: {size_gb:.2f} GB")
print("Expected: ~0.79 GB (1.5B x 0.5 bytes + ~35 MB of group scales).")
print("Matches the Day 30 GPTQ checkpoint size — same W4A16 footprint.")

## Wrap-up — record your numbers

Copy these into the "What to measure" table in the Day 31 PDF:

| Metric | Your number |
|---|---|
| AWQ 4-bit g128 perplexity (WikiText-2 test) | |
| Delta ppl AWQ vs fp16 (target: <= GPTQ's delta) | |
| AWQ quantization wall time (T4) | |
| Checkpoint size on disk (expect ~0.79 GB) | |
| GPTQ-vs-AWQ delta ppl difference | |
| 50-token generation check (coherent? y/n) | |

Then write your one-paragraph GPTQ-vs-AWQ decision: which would you ship on
your own serving stack, and why?